# ¿Es no local la materia oscura aparente? La RAR con puentes ER

**Proyecto:** *Vacuum Particle Journey*, cuarta parte.

**Lo que dijo el notebook 3:** los datos piden la forma exacta de la relación de aceleración radial (RAR) de McGaugh. Nuestra ley de potencias κ(g) quedaba muy por detrás (ΔBIC ≈ 64 000) y dejaba residuos en forma de U. En galaxias, lo único que la red ER puede añadir a la RAR es **no localidad**: que la masa extra se reparta a escala λ en lugar de actuar punto a punto.

**Modelo de este notebook:**
1. Se calcula la "masa fantasma" que la RAR necesita en cada radio:
$$G M_{ph}(r) = r^2\big[g_{RAR}(g_b) - g_b\big],\qquad g_{RAR} = \frac{g_b}{1-e^{-\sqrt{g_b/g_\dagger}}}$$
2. Esa masa se reparte con el núcleo de los puentes ER, $K_\lambda(d) = e^{-d/\lambda}/(4\pi\lambda^2 d)$.
3. $v^2 = v_b^2 + G M_{ph,\lambda}(r)/r$.

**Con λ = 0 es exactamente la RAR.** Así que la pregunta es limpia: *¿prefieren los datos λ > 0?*
- **Sí, y de forma robusta:** hay una señal que MOND y la RAR no tienen, y que la red ER explica.
- **No (λ → 0):** en galaxias, la red ER es indistinguible de la RAR, y hay que buscar la diferencia en cúmulos o lentes.

**Cómo se evita engañarse:**
- **Perfil Δχ²(λ):** cuánto mejora el ajuste al encender λ.
- **Bootstrap de galaxias:** se remuestrean las galaxias 1000 veces para ver si λ > 0 depende de unas pocas.
- **Validación cruzada:** ¿predice mejor la versión no local en galaxias que no ha visto?
- **Extrapolación radial:** ajustar el interior y predecir el exterior.
- **Robustez:** se repite todo con un 5 % de error sistemático añadido. Los errores de SPARC no incluyen bien la distancia ni la inclinación, y una señal que solo exista con los errores originales sería sospechosa.
- **¿Es λ universal?** Si cada galaxia prefiere un λ proporcional a su tamaño, no es una constante física.

> Tiempo en Colab: ~3–6 min.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2

## 1. Datos SPARC (igual que en los notebooks anteriores)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

## 2. El modelo RAR no local

Para que el límite λ → 0 sea exacto, el reparto se hace analíticamente. Un cascarón de masa en el radio $a$, repartido con el núcleo $K_\lambda$, deja dentro del radio $r$ una fracción $F_\lambda(r,a)$ con forma cerrada (con $x = r/\lambda$):

$$F_\lambda(r,a) = \frac{\lambda}{2a}\times\begin{cases} e^{-(a-r)/\lambda}(x-1) + e^{-(a+r)/\lambda}(1+x) & r\le a\\[2pt] \dfrac{2a}{\lambda} - e^{-(r-a)/\lambda}(1+x) + e^{-(a+r)/\lambda}(1+x) & r>a\end{cases}$$

Cuando λ → 0, $F$ se convierte en un escalón (el cascarón está dentro o fuera), y se recupera la RAR local. La masa fantasma se discretiza en 400 cascarones por galaxia.

In [ ]:
NG = 400
U_GRID = np.geomspace(0.1, 1.5, 40)
LAM_GRID = np.r_[0.0, np.geomspace(0.1, 100, 24)]                                       # kpc
GD_GRID = np.unique(np.r_[np.geomspace(0.6e-10, 2.4e-10, 25), 1.2e-10]) / TO_MS2        # (km/s)^2/kpc
I_GD12 = int(np.argmin(np.abs(GD_GRID * TO_MS2 - 1.2e-10)))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def rar_g(gb, gd):
    return gb / (-np.expm1(-np.sqrt(gb / gd)))

def F_matrix(r, a, lam):
    R, A = r[:, None], a[None, :]
    if lam == 0:
        return (A < R).astype(float)
    x = R / lam
    e_lo = np.exp(-np.clip((A - R) / lam, 0, None))
    e_hi = np.exp(-np.clip((R - A) / lam, 0, None))
    e_sum = np.exp(-(A + R) / lam)
    I_lo = e_lo * (x - 1) + e_sum * (1 + x)
    I_hi = 2 * A / lam - e_hi * (1 + x) + e_sum * (1 + x)
    return np.where(R <= A, I_lo, I_hi) * lam / (2 * A)

def setup(g):
    if "_s" in g:
        return g["_s"]
    rmax = g["r"].max()
    rg = np.linspace(rmax / NG, rmax, NG)
    s = dict(rg=rg, mids=0.5 * (np.r_[0.0, rg[:-1]] + rg),
             vg=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vgas2"]]),
             vs=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vstar2"]]))
    g["_s"] = s
    return s

def phantom_shells(s, U, GD):
    # G·M_fantasma en cascarones -> forma (nΥ, ng†, NG)
    v2 = s["vg"][None, :] + U[:, None] * s["vs"][None, :]
    gb = np.clip(v2, 1e-6, None) / s["rg"][None, :]
    gr = rar_g(gb[:, None, :], GD[None, :, None])
    GM = s["rg"][None, None, :] ** 2 * (gr - gb[:, None, :])
    return np.diff(np.concatenate([np.zeros(GM.shape[:2] + (1,)), GM], axis=2), axis=2)

def model_v2(g, u, lam, gd):
    s = setup(g)
    d = phantom_shells(s, np.array([u]), np.array([gd]))[0, 0]
    return vb2(g, u) + F_matrix(g["r"], s["mids"], lam) @ d / g["r"]

# Comprobaciones: F(λ→0) es un escalón; con λ pequeño se recupera la RAR local exacta
_g = next(iter(GAL.values()))
v_loc = np.sqrt(_g["r"] * rar_g(np.clip(vb2(_g, 0.5), 1e-6, None) / _g["r"], 1.2e-10 / TO_MS2))
v_0 = np.sqrt(model_v2(_g, 0.5, 0.0, 1.2e-10 / TO_MS2))
v_s = np.sqrt(model_v2(_g, 0.5, 1e-3, 1.2e-10 / TO_MS2))
print(f"Test: |v(λ=0) − RAR exacta| máx = {np.max(np.abs(v_0 - v_loc)):.2f} km/s; "
      f"|v(λ=0.001) − v(λ=0)| máx = {np.max(np.abs(v_s - v_0)):.3f} km/s (ambos deben ser ≈ 0)")

## 3. Tabla de χ² por galaxia

Para cada galaxia, y para cada par $(\lambda, g_\dagger)$ de la rejilla, se busca el mejor Υ y se guarda el χ². Con esa tabla, **cualquier ajuste global** (muestra completa, subgrupo de validación cruzada o remuestreo bootstrap) es solo una suma de filas, así que se puede repetir 1000 veces al instante.

Se calculan dos tablas: con los errores originales de SPARC y con un 5 % de error sistemático añadido en cuadratura.

In [ ]:
def chi_table(galaxies, floor=0.0):
    names = list(galaxies)
    TOT = np.zeros((len(names), len(LAM_GRID), len(GD_GRID)))
    DAT = np.zeros_like(TOT); UB = np.zeros_like(TOT)
    for n_i, name in enumerate(names):
        g = galaxies[name]; s = setup(g)
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        d = phantom_shells(s, U_GRID, GD_GRID)                          # u, gd, NG
        base = vb2(g, U_GRID[:, None])                                   # u, N
        pr = prior(U_GRID) ** 2
        for l_i, lam in enumerate(LAM_GRID):
            extra = np.einsum("in,ugn->ugi", F_matrix(g["r"], s["mids"], lam), d) / g["r"]
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = (((g["vobs"] - v) / err) ** 2).sum(-1)                  # u, gd
            t = c + pr[:, None]
            k = t.argmin(0)
            TOT[n_i, l_i] = t[k, np.arange(len(GD_GRID))]
            DAT[n_i, l_i] = c[k, np.arange(len(GD_GRID))]
            UB[n_i, l_i] = U_GRID[k]
    return dict(names=names, tot=TOT, dat=DAT, ups=UB)

t0 = time.time()
TAB = chi_table(GAL, 0.0)
print(f"Tabla con errores originales: {time.time()-t0:.0f} s")
t0 = time.time()
TAB5 = chi_table(GAL, 0.05)
print(f"Tabla con 5 % sistemático: {time.time()-t0:.0f} s")

## 4. Ajuste global y perfil Δχ²(λ)

El perfil muestra cuánto empeora el χ² total al fijar λ, con $g_\dagger$ optimizado en cada caso. La referencia es λ = 0 (RAR local).
- Si la curva tiene su mínimo en λ > 0 y la caída respecto a λ = 0 supera $\ln N$ (la penalización BIC de un parámetro), los datos prefieren la no localidad.

In [ ]:
def global_best(tab, idx=None, lam_fixed=None):
    T = tab["tot"] if idx is None else tab["tot"][idx]
    S = T.sum(0)
    if lam_fixed is not None:
        j = int(np.argmin(S[lam_fixed])); return lam_fixed, j, S[lam_fixed, j]
    i, j = np.unravel_index(np.argmin(S), S.shape)
    return i, j, S[i, j]

def report(tab, label):
    S = tab["tot"].sum(0)
    i, j, best = global_best(tab)
    i0_, j0, best0 = global_best(tab, lam_fixed=0)
    prof = S.min(1) - best
    print(f"[{label}] mejor: λ = {LAM_GRID[i]:.3g} kpc, g† = {GD_GRID[j]*TO_MS2:.3g} m/s² | "
          f"local: g† = {GD_GRID[j0]*TO_MS2:.3g} | mejora Δχ² (local − no local) = {best0 - best:.1f}")
    return prof, (i, j), (j0,)

PROF, BEST, BEST0 = report(TAB, "errores originales")
PROF5, BEST5, BEST05 = report(TAB5, "con 5 % sistemático")

N_TOT_PTS = sum(len(g["r"]) for g in GAL.values())
x = np.r_[0.05, LAM_GRID[1:]]
fig, ax = plt.subplots(figsize=(8, 4.2))
for prof, lab, c in [(PROF, "Errores originales", COL["RAR no local (ER)"]),
                     (PROF5, "Con 5 % sistemático", COL["RAR local (g† global)"])]:
    ax.semilogx(x, np.clip(prof, 0.1, None), "o-", ms=4, color=c, label=lab)
ax.axhline(np.log(N_TOT_PTS), color="#9a9a95", ls="--", lw=1, label=f"Penalización BIC de 1 parámetro (ln N = {np.log(N_TOT_PTS):.1f})")
ax.set_yscale("log")
ax.set_xticks([0.05, 0.1, 1, 10, 100]); ax.set_xticklabels(["0\n(RAR local)", "0.1", "1", "10", "100"])
ax.set_xlabel("λ, alcance de los puentes ER [kpc]"); ax.set_ylabel("Δχ² respecto al mejor λ")
ax.set_title("Perfil de verosimilitud del alcance λ", loc="left"); ax.legend(fontsize=8.5)
plt.show()

## 5. Prueba 1: BIC total

Se añaden NFW libre y NFW ΛCDM como referencia (se ajustan galaxia a galaxia por mínimos cuadrados).

In [ ]:
def _v2_nfw_(r, V200, c):
    x = r / (V200 / (10 * H0_KPC)); mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def c_dutton(V200):
    M200 = V200**3 / (10 * G * H0_KPC)
    return 10 ** (0.905 - 0.101 * np.log10(M200 / (1e12 / (H0 / 100))))

def fit_nfw(g, cm=True, floor=0.0):
    err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
    if cm:
        f = lambda p: vb2(g, p[0]) + _v2_nfw_(g["r"], p[1], c_dutton(p[1])); lo, hi, p0 = [0.1, 10], [1.5, 500], [0.5, 100]
    else:
        f = lambda p: vb2(g, p[0]) + _v2_nfw_(g["r"], p[1], p[2]); lo, hi, p0 = [0.1, 10, 1], [1.5, 500, 50], [0.5, 100, 10]
    best = None
    for start in [p0] + [list(np.array(lo) + RNG.random(len(lo)) * (np.array(hi) - np.array(lo))) for _ in range(3)]:
        r = least_squares(lambda p: np.r_[(g["vobs"] - np.sqrt(np.clip(f(p), 0, None))) / err, prior(p[0])],
                          start, bounds=(lo, hi))
        best = r if best is None or r.cost < best.cost else best
    v = np.sqrt(np.clip(f(best.x), 0, None))
    return dict(chi2=float((((g["vobs"] - v) / err) ** 2).sum()), k=len(best.x), v=v, p=best.x)

t0 = time.time()
NFW = {n: {"NFW ΛCDM (c–M)": fit_nfw(g, True), "NFW libre": fit_nfw(g, False)} for n, g in GAL.items()}
print(f"NFW ajustado en {time.time()-t0:.0f} s")

def bic_table(tab, nfw=None):
    i, j = BEST if tab is TAB else BEST5
    j0 = (BEST0 if tab is TAB else BEST05)[0]
    rows = []
    def add(name, chi2, kgal, shared, ups=None):
        K = kgal * len(GAL) + shared
        rows.append(dict(Modelo=name, parametros=K, chi2_total=round(chi2), BIC=round(chi2 + K * np.log(N_TOT_PTS)),
                         Υ_mediana=None if ups is None else round(float(np.median(ups)), 2)))
    add("RAR (g† = 1.2e-10)", tab["dat"][:, 0, I_GD12].sum(), 1, 0, tab["ups"][:, 0, I_GD12])
    add("RAR local (g† global)", tab["dat"][:, 0, j0].sum(), 1, 1, tab["ups"][:, 0, j0])
    add("RAR no local (ER)", tab["dat"][:, i, j].sum(), 1, 2, tab["ups"][:, i, j])
    if nfw is not None:
        for m, kg in [("NFW ΛCDM (c–M)", 2), ("NFW libre", 3)]:
            add(m, sum(nfw[n][m]["chi2"] for n in GAL), kg, 0)
    d = pd.DataFrame(rows).set_index("Modelo"); d["ΔBIC"] = d.BIC - d.BIC.min()
    return d

print("Errores originales:"); display_tab = bic_table(TAB, NFW); print(display_tab.to_string())
print("\nCon 5 % sistemático (sin NFW):"); print(bic_table(TAB5).to_string())

## 6. Prueba 2: bootstrap de galaxias

Se remuestrean las galaxias (con reemplazo) 1000 veces y en cada caso se busca el mejor λ. Si la preferencia por λ > 0 depende de unas pocas galaxias, muchas réplicas darán λ = 0.

In [ ]:
def boot(tab, n=1000):
    lams = np.empty(n); gain = np.empty(n)
    N = len(tab["names"])
    for b in range(n):
        idx = RNG.integers(0, N, N)
        i, j, best = global_best(tab, idx)
        _, _, best0 = global_best(tab, idx, lam_fixed=0)
        lams[b] = LAM_GRID[i]; gain[b] = best0 - best
    return lams, gain

BL, BG = boot(TAB); BL5, BG5 = boot(TAB5)
for lab, L_, G_ in [("errores originales", BL, BG), ("5 % sistemático", BL5, BG5)]:
    lo, hi = np.percentile(L_, [2.5, 97.5])
    print(f"[{lab}] λ = 0 en {np.mean(L_ == 0):.0%} de réplicas | IC 95 % de λ: {lo:.3g} – {hi:.3g} kpc | "
          f"Δχ² mediana {np.median(G_):.1f}")

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.r_[0, np.geomspace(0.08, 120, 26)]
ax.hist(np.where(BL == 0, 0.05, BL), bins=np.r_[0.04, bins[1:]], histtype="step", lw=2, color=COL["RAR no local (ER)"], label="Errores originales")
ax.hist(np.where(BL5 == 0, 0.05, BL5), bins=np.r_[0.04, bins[1:]], histtype="step", lw=2, color=COL["RAR local (g† global)"], label="Con 5 % sistemático")
ax.set_xscale("log"); ax.set_xlabel("λ preferido en cada réplica [kpc]  (la barra de 0.05 = λ = 0)")
ax.set_ylabel("Réplicas"); ax.set_title("Bootstrap de galaxias", loc="left"); ax.legend(fontsize=9)
plt.show()

## 7. Prueba 3: validación cruzada (la más importante)

Se aprenden $(\lambda, g_\dagger)$ con el 80 % de las galaxias, y la RAR local aprende su $g_\dagger$ con las mismas. Luego se comparan las dos **en el 20 % que no han visto**. Si la versión no local predice mejor fuera de muestra de forma sistemática, la no localidad es real y no un ajuste al ruido.

In [ ]:
def cv(tab, k=5, reps=4):
    out = []
    N = len(tab["names"])
    for rep in range(reps):
        perm = RNG.permutation(N)
        for f, test in enumerate(np.array_split(perm, k)):
            train = np.setdiff1d(perm, test)
            i, j, _ = global_best(tab, train)
            _, j0, _ = global_best(tab, train, lam_fixed=0)
            out.append(dict(rep=rep, grupo=f, lam=LAM_GRID[i],
                            chi_local=tab["tot"][test, 0, j0].sum(), chi_nolocal=tab["tot"][test, i, j].sum()))
    d = pd.DataFrame(out); d["mejora"] = d.chi_local - d.chi_nolocal
    return d

CV = cv(TAB); CV5 = cv(TAB5)
for lab, d in [("errores originales", CV), ("5 % sistemático", CV5)]:
    print(f"[{lab}] grupos donde la versión no local predice mejor: {np.mean(d.mejora > 0):.0%} | "
          f"mejora total Δχ² fuera de muestra = {d.mejora.sum()/4:.0f} (por repetición) | λ aprendido: "
          f"{d.lam.min():.3g}–{d.lam.max():.3g} kpc")

## 8. Prueba 4: extrapolación radial

Con $(\lambda, g_\dagger)$ globales, se ajusta Υ solo con el 60 % interior de cada curva y se mide el χ²ν del 40 % exterior.

In [ ]:
def fit_u(g, lam, gd, mask, floor=0.0):
    err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
    def f(lu):
        v = np.sqrt(np.clip(model_v2(g, 10 ** lu, lam, gd), 0, None))
        return float((((g["vobs"] - v) / err)[mask] ** 2).sum()) + prior(10 ** lu) ** 2
    r = minimize_scalar(f, bounds=(-1, np.log10(1.5)), method="bounded", options=dict(xatol=1e-3))
    u = 10 ** r.x
    return u, np.sqrt(np.clip(model_v2(g, u, lam, gd), 0, None))

lam_b, gd_b = LAM_GRID[BEST[0]], GD_GRID[BEST[1]]; gd_0 = GD_GRID[BEST0[0]]
ext = []
for name, g in GAL.items():
    N = len(g["r"])
    if N < 10:
        continue
    tr = np.zeros(N, bool); tr[: int(0.6 * N)] = True; te = ~tr
    for m, (lam, gd) in {"RAR local (g† global)": (0.0, gd_0), "RAR no local (ER)": (lam_b, gd_b)}.items():
        _, v = fit_u(g, lam, gd, tr)
        ext.append(dict(galaxia=name, modelo=m, exterior=float((((g["vobs"] - v) / g["err"])[te] ** 2).mean())))
EXT = pd.DataFrame(ext)
piv = EXT.pivot(index="galaxia", columns="modelo", values="exterior")
print(EXT.groupby("modelo").exterior.median().round(2).to_string())
print(f"Galaxias donde la versión no local predice mejor el exterior: "
      f"{np.mean(piv['RAR no local (ER)'] < piv['RAR local (g† global)']):.0%} de {len(piv)}")

## 9. ¿Es λ una constante universal?

Para cada galaxia buscamos su λ preferido (con el $g_\dagger$ global) y cuánto mejora sobre λ = 0. Si λ crece con el tamaño de la galaxia, no es una propiedad de los puentes ER sino un artefacto (por ejemplo, de cómo se distribuye el gas). Una constante física no debería depender del tamaño.

In [ ]:
j = BEST[1]
per = []
for n_i, name in enumerate(TAB["names"]):
    c = TAB["tot"][n_i, :, j]
    per.append(dict(galaxia=name, lam=LAM_GRID[int(np.argmin(c))], mejora=c[0] - c.min(), rmax=GAL[name]["r"].max()))
PER = pd.DataFrame(per)
sig = PER.mejora > 4
print(f"Galaxias que prefieren λ > 0 con Δχ² > 4: {sig.mean():.0%} | λ = 0 preferido: {np.mean(PER.lam == 0):.0%}")
if sig.sum() > 5:
    rho, p = spearmanr(PER.loc[sig, "rmax"], PER.loc[sig, "lam"])
    print(f"Correlación λ preferido – tamaño de la galaxia (Spearman): ρ = {rho:.2f}, p = {p:.1e}")
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(PER.rmax[~sig], np.where(PER.lam[~sig] == 0, 0.05, PER.lam[~sig]), "o", ms=4, color="#c3c2b7", label="Sin preferencia clara")
ax.loglog(PER.rmax[sig], np.where(PER.lam[sig] == 0, 0.05, PER.lam[sig]), "o", ms=5, color=COL["RAR no local (ER)"], label="Prefiere λ (Δχ² > 4)")
ax.axhline(lam_b if lam_b > 0 else 0.05, color="#222222", ls="--", lw=1, label="λ global")
ax.set_xlabel("Radio máximo medido [kpc]"); ax.set_ylabel("λ preferido [kpc]  (0.05 = λ = 0)")
ax.set_title("¿Depende λ del tamaño de la galaxia?", loc="left"); ax.legend(fontsize=8.5)
plt.show()

## 10. NGC 3198, NGC 2403 y DDO 154

In [ ]:
def detail(name):
    if name not in GAL:
        print(name, "no está en la muestra"); return
    g = GAL[name]; r = g["r"]; full = np.ones(len(r), bool)
    curves = {"RAR (g† = 1.2e-10)": fit_u(g, 0.0, 1.2e-10 / TO_MS2, full),
              "RAR local (g† global)": fit_u(g, 0.0, gd_0, full),
              "RAR no local (ER)": fit_u(g, lam_b, gd_b, full)}
    fig, (ax, axr) = plt.subplots(2, 1, figsize=(8.5, 6.5), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.2], hspace=0.06))
    ax.errorbar(r, g["vobs"], g["err"], fmt="o", ms=4, color=COL["Datos"], ecolor="#9a9a95", elinewidth=1, label="Observado", zorder=5)
    ax.plot(r, np.sqrt(np.clip(vb2(g, 0.5), 0, None)), "--", color=COL["Solo bariones"], label="Solo bariones (Υ=0.5)")
    rows = []
    for m, (u, v) in list(curves.items()) + [("NFW ΛCDM (c–M)", (NFW[name]["NFW ΛCDM (c–M)"]["p"][0], NFW[name]["NFW ΛCDM (c–M)"]["v"]))]:
        c2 = float((((g["vobs"] - v) / g["err"]) ** 2).sum()); k = 2 if "NFW" in m else 1
        ax.plot(r, v, color=COL[m], label=f"{m} (χ²ν={c2/max(len(r)-k,1):.1f})")
        axr.plot(r, (g["vobs"] - v) / g["err"], "o-", ms=2.5, lw=1, color=COL[m])
        rows.append(dict(Modelo=m, chi2r=round(c2 / max(len(r) - k, 1), 2), Υ=round(float(u), 3)))
    ax.set_ylim(0, None); ax.set_ylabel("Velocidad [km/s]"); ax.set_title(name, loc="left"); ax.legend(fontsize=8.5, loc="lower right")
    axr.axhline(0, color="#c3c2b7", lw=1); axr.set_ylabel("Residuo [σ]"); axr.set_xlabel("Radio [kpc]")
    plt.show()
    return pd.DataFrame(rows).set_index("Modelo")

detail("NGC3198")

In [ ]:
detail("NGC2403")

In [ ]:
detail("DDO154")

## 11. Veredicto automático

Para hablar de **evidencia de no localidad** se exigen las cuatro condiciones a la vez:
1. ΔBIC (RAR local − no local) > 10, **también con el 5 % sistemático**;
2. menos del 5 % de réplicas bootstrap con λ = 0;
3. mejor predicción fuera de muestra en la mayoría de los grupos de validación cruzada;
4. que λ no dependa claramente del tamaño de la galaxia.

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS (con halo NFW): solo comprueba que el código funciona.\n")
    b = bic_table(TAB); b5 = bic_table(TAB5)
    d1 = b.loc["RAR local (g† global)", "BIC"] - b.loc["RAR no local (ER)", "BIC"]
    d5 = b5.loc["RAR local (g† global)", "BIC"] - b5.loc["RAR no local (ER)", "BIC"]
    c1 = d1 > 10 and d5 > 10
    c2 = np.mean(BL == 0) < 0.05 and np.mean(BL5 == 0) < 0.05
    c3 = np.mean(CV.mejora > 0) > 0.7 and np.mean(CV5.mejora > 0) > 0.7
    c4 = True
    if sig.sum() > 5:
        rho, p = spearmanr(PER.loc[sig, "rmax"], PER.loc[sig, "lam"]); c4 = not (rho > 0.5 and p < 0.01)
    print(f"1) ΔBIC a favor de no local: {d1:+.0f} (errores originales), {d5:+.0f} (5 % sistemático) → {'✓' if c1 else '✗'}")
    print(f"2) Réplicas bootstrap con λ = 0: {np.mean(BL==0):.0%} / {np.mean(BL5==0):.0%} → {'✓' if c2 else '✗'}")
    print(f"3) Grupos de validación cruzada donde no local predice mejor: {np.mean(CV.mejora>0):.0%} / {np.mean(CV5.mejora>0):.0%} → {'✓' if c3 else '✗'}")
    print(f"4) λ independiente del tamaño de la galaxia → {'✓' if c4 else '✗'}")
    print(f"\nλ global = {lam_b:.3g} kpc, g† = {gd_b*TO_MS2:.3g} m/s²")
    if c1 and c2 and c3 and c4:
        print("\n→ EVIDENCIA DE NO LOCALIDAD: los datos prefieren que la masa extra se reparta a escala λ. "
              "Es una señal que la RAR/MOND no tienen y que la red ER predice. Hay que confirmarla con otros datos (lentes, otras muestras).")
    elif c1 or c3:
        print("\n→ INDICIOS, NO EVIDENCIA: alguna prueba favorece λ > 0, pero no todas. Puede ser una señal o un efecto sistemático de los datos.")
    else:
        print("\n→ SIN SEÑAL: en galaxias la red ER no se distingue de la RAR local. La diferencia hay que buscarla en cúmulos, lentes o cosmología.")

veredicto()

## 12. Cómo leer el resultado

- **Si hay evidencia de no localidad:** es lo primero de todo el proyecto que la RAR/MOND no explica y la red ER sí. El valor de λ se convierte en una predicción para otros sistemas (lentes débiles alrededor de galaxias, galaxias con anillos o bordes de disco bruscos).
- **Si no hay señal:** no refuta la hipótesis ER, pero dice que en galaxias se comporta exactamente como la RAR. La interpretación (entrelazamiento que sobrevive donde hay poca decoherencia) sigue siendo una explicación posible *de* la RAR, y las pruebas que pueden distinguirla son los cúmulos y la cosmología.
- **Precaución:** en ambos casos, χ²ν > 1 indica que los errores de SPARC no recogen todos los sistemáticos. Por eso se exige que la señal sobreviva al 5 % de error añadido.